# Quant Backtest — Analysis
Equity curves, drawdowns, rolling metrics, and portfolio weights.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from data import load, get_close
from features import build_feature_matrix
from strategy import momentum_strategy, composite_strategy, risk_managed_strategy
from backtest import run, BacktestConfig, benchmark_returns, benchmark_equity
from metrics import summary, compare, drawdown_series, drawdown_table

sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 110

COLORS = {'Momentum': '#2196F3', 'Composite': '#FF9800', 'Risk-Managed': '#4CAF50', 'SPY': '#9E9E9E'}

In [ ]:
close = get_close(load())
fm    = build_feature_matrix(close)
cfg   = BacktestConfig(commission=0.001, slippage=0.0005)

w1 = momentum_strategy(fm, top_n=10, signal='mom_6m')
w2 = composite_strategy(fm, top_n=10)
w3 = risk_managed_strategy(fm, close, top_n=10, target_vol=0.15)

r1 = run(w1, close, cfg)
r2 = run(w2, close, cfg)
r3 = run(w3, close, cfg)

bm_eq  = benchmark_equity(close).reindex(r1['equity'].index)
bm_ret = bm_eq.pct_change().fillna(0)

def norm(s): return s / s.iloc[0] * 100

eq = pd.DataFrame({
    'Momentum':     norm(r1['equity']),
    'Composite':    norm(r2['equity']),
    'Risk-Managed': norm(r3['equity']),
    'SPY':          norm(bm_eq),
})
print('Date range:', eq.index[0].date(), '->', eq.index[-1].date())

## 1. Equity Curves

In [ ]:
fig, ax = plt.subplots()
for col in eq.columns:
    ax.plot(eq.index, eq[col], label=col,
            color=COLORS[col],
            linestyle='--' if col == 'SPY' else '-',
            linewidth=1.2 if col == 'SPY' else 1.8)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}'))
ax.set_title('Equity Curves (log scale, base=100)', fontsize=13, fontweight='bold')
ax.set_ylabel('Portfolio Value')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Drawdown Comparison

In [ ]:
dd = pd.DataFrame({
    'Momentum':     drawdown_series(r1['equity']),
    'Composite':    drawdown_series(r2['equity']),
    'Risk-Managed': drawdown_series(r3['equity']),
    'SPY':          drawdown_series(bm_eq),
})
fig, ax = plt.subplots()
for col in dd.columns:
    ax.fill_between(dd.index, dd[col] * 100, 0, alpha=0.3, color=COLORS[col], label=col)
    ax.plot(dd.index, dd[col] * 100, color=COLORS[col], linewidth=0.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Drawdown (%)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Summary Table

In [ ]:
spy_ret = benchmark_returns(close, 'SPY').reindex(r1['returns'].index)
report = compare([
    summary(r1['equity'], r1['returns'], spy_ret, label='Momentum'),
    summary(r2['equity'], r2['returns'], spy_ret, label='Composite'),
    summary(r3['equity'], r3['returns'], spy_ret, label='Risk-Managed'),
    summary(bm_eq, bm_ret, label='SPY'),
])
report

## 4. Rolling Sharpe (252-day)

In [ ]:
def rolling_sharpe(returns, window=252, rf=0.04):
    rf_d = (1 + rf) ** (1/252) - 1
    exc  = returns - rf_d
    return (exc.rolling(window).mean() / exc.rolling(window).std()) * np.sqrt(252)

fig, ax = plt.subplots()
for label, ret in [('Momentum', r1['returns']), ('Composite', r2['returns']),
                   ('Risk-Managed', r3['returns']), ('SPY', bm_ret)]:
    ax.plot(rolling_sharpe(ret), label=label, color=COLORS[label],
            linewidth=1.2 if label == 'SPY' else 1.6,
            linestyle='--' if label == 'SPY' else '-')
ax.axhline(0, color='black', linewidth=0.6, linestyle=':')
ax.axhline(1, color='green', linewidth=0.6, linestyle=':')
ax.set_title('Rolling 252-day Sharpe Ratio', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Monthly Returns Heatmap (Risk-Managed)

In [ ]:
monthly = (1 + r3['returns']).resample('ME').prod() - 1
table   = monthly.groupby([monthly.index.year, monthly.index.month]).first().unstack()
table.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(14, len(table) * 0.55 + 1))
sns.heatmap(table * 100, annot=True, fmt='.1f', center=0,
            cmap='RdYlGn', linewidths=0.4, ax=ax,
            cbar_kws={'label': 'Monthly Return (%)'})
ax.set_title('Monthly Returns — Risk-Managed (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

## 6. Turnover & Costs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, res in [('Momentum', r1), ('Composite', r2), ('Risk-Managed', r3)]:
    axes[0].plot(res['turnover'].resample('ME').sum(), label=label, color=COLORS[label], linewidth=1.4)
    axes[1].plot(res['costs'].resample('ME').sum() * 1e4, label=label, color=COLORS[label], linewidth=1.4)
axes[0].set_title('Monthly One-Way Turnover', fontweight='bold')
axes[0].legend()
axes[1].set_title('Monthly Transaction Costs (bps)', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Drawdown Periods Table

In [ ]:
print('=== Composite ===')
display(drawdown_table(r2['equity'], top_n=5))
print('\n=== Risk-Managed ===')
display(drawdown_table(r3['equity'], top_n=5))

## 8. Current Portfolio Weights (Risk-Managed)

In [ ]:
last_w = r3['weights_act'].iloc[-1].sort_values(ascending=False)
last_w = last_w[last_w > 0.001]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(last_w.index, last_w * 100, color='#4CAF50', edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(f'Current Weights — Risk-Managed ({r3["weights_act"].index[-1].date()})',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Weight (%)')
plt.tight_layout()
plt.show()